**STEP 0: Install the library prerequisites**

---



In [ ]:
%pip3 install janus_swi ollama openai

**STEP 1.0: Define a system prompt, few-shot examples, and dynamic prompt generation function**

In [1]:
SYSTEM_PROMPT = r"""
### Role
You are an expert Prolog programmer specializing in Knowledge Representation and Reasoning for GraphRAG systems.

### Task
Translate the provided Context into a formal Prolog database to answer the User Question. You must not conclude anything for yourself; only generate the Prolog code that could be used to answer the question.

### Execution Flow (Mandatory)
You must follow this exact sequence for every response:
1. For every logical leap, use a SEPARATE <reasoning_step> tag.
2. Inside <reasoning_step>, explain the logic in English, then provide the Prolog code snippet.
3. You MUST close every </reasoning_step> before starting the next one.
4. After all reasoning is complete, provide the full <database>. The database must be enclosed within <database> and </database> tags.
5. Finally, provide the <query>. The query must be enclosed within <query> and </query> tags.
6. You MUST close the </database> and </query> tags.

### Syntactic Requirements
- All Prolog atoms (facts/predicates) must be lowercase.
- Use CamelCase for Variables (e.g., PersonName, UncleObject).
- Every Prolog line must end with a period (.).
- No single-letter variables (Use 'Parent' instead of 'P').

### Structural Output Format (Strict)
Your output must strictly follow this visual template:

<reasoning_step>
[Reasoning for facts or rules or queries]
[Prolog facts or rules or queries]
</reasoning_step>

<reasoning_step>
[Reasoning for facts or rules or queries]
[Prolog facts or rules or queries]
</reasoning_step>

<database>
[All facts and rules combined here. NO natural language inside these tags.]
</database>

<query>
[The specific query only. NO natural language inside these tags.]
</query>

---

### Final Constraint
It is important that you enclose the query within <query> and </query> tags. 
"""

few_shot_examples = [
    # Example 1: Brother or Amanda.
    {"role": "user", "content": """
Now, process the following:
Context: Bob is Alice's brother. Amanda is Alice's sister.
Query: Who is Alice's brother?
"""},
    {"role": "assistant", "content": r"""
<reasoning_step>
The context identifies Bob as a male, Alice as a female, and Amanda as a female. They share a sibling relationship via a common parent (p1).
male(bob).
female(alice).
female(amanda).
parent(p1, bob).
parent(p1, alice).
parent(p1, amanda).
</reasoning_step>

<reasoning_step>
To answer "who is a brother of Amanda," I need a rule defining a brother as a male sibling.
sibling(Sibling1, Sibling2) :- parent(Parent, Sibling1), parent(Parent, Sibling2), Sibling1 \= Sibling2.
brother(Brother, Sibling) :- male(Brother), sibling(Brother, Sibling).
</reasoning_step>


<reasoning_step>
The question "who is a brother of Amanda," requires a variable as it implies a query to find all Brothers such that Brother is a brother of Amanda. 
brother(Brother, amanda).
</reasoning_step>

<database>
male(bob).
female(alice).
female(amanda).
parent(p1, bob).
parent(p1, alice).
parent(p1, amanda).
sibling(Sibling1, Sibling2) :- parent(Parent, Sibling1), parent(Parent, Sibling2), Sibling1 \= Sibling2.
brother(Brother, Sibling) :- male(Brother), sibling(Brother, Sibling).
</database>

<query>
brother(Brother, amanda).
</query>

---
"""},
    
    # Example 2: One-hop reasoning
    {"role": "user", "content": """
Now, process the following:
Context: Maria is the mother of Chloe. Chloe is the mother of Dan. 
Query: Is Maria the grandmother of Dan?
"""},
    {"role": "assistant", "content": r"""
<reasoning_step> 
The context identifies Maria as Chloe's mother and Chloe as Dan's mother. I will use lowercase atoms for names. 
parent(maria, chloe). 
parent(chloe, dan). 
</reasoning_step>

<reasoning_step> 
To answer the grandmother query, I need a rule defining a grandmother as a parent of a parent. 
grandmother(Grandmother, Grandchild) :- parent(Grandmother, IntermediateParent), parent(IntermediateParent, Grandchild). 
</reasoning_step>

<reasoning_step> 
The question does not require a variable; it is a yes/no question about whether Maria is the grandmother of Dan. 
grandmother(maria, dan). 
</reasoning_step>

<database> 
parent(maria, chloe). 
parent(chloe, dan). 
grandmother(Grandmother, Grandchild) :- parent(Grandmother, IntermediateParent), parent(IntermediateParent, Grandchild). 
</database>

<query> 
grandmother(maria, dan). 
</query>

---
"""},
    
    # Example 2: One-hop reasoning
    {"role": "user", "content": """
Now, process the following:
Context: Tomasinno is the friend of Vito. Solozzo is the enemy of Vito. Vito believes that he should keep his enemies closer than his friends. 
Query: Who does Vito believe he should keep closer?
"""},
    {"role": "assistant", "content": r"""
<reasoning_step> 
The context identifies Tomasinno as Vito's friend, Solozzo as Vito's enemy, and Vito believes enemies should be kept closer than friends. I will use lowercase atoms for names. 
friend(tomasinno, vito). 
enemy(solozzo, vito). 
belief(vito, keep_enemies_closer_than_friends). 
</reasoning_step>

<reasoning_step> 
To answer the grandmother query, I need a rule defining a grandmother as a parent of a parent. 
grandmother(Grandmother, Grandchild) :- parent(Grandmother, IntermediateParent), parent(IntermediateParent, Grandchild). 
</reasoning_step>

<reasoning_step> 
The question does not require a variable; it is a yes/no question about whether Maria is the grandmother of Dan. 
grandmother(maria, dan). 
</reasoning_step>

<database> 
parent(maria, chloe). 
parent(chloe, dan). 
grandmother(Grandmother, Grandchild) :- parent(Grandmother, IntermediateParent), parent(IntermediateParent, Grandchild). 
</database>

<query> 
grandmother(maria, dan). 
</query>

---
"""},
    
    # Example 3: Simple Educational QA
    {"role": "user", "content": """
Now, process the following:
Context: A square is a polygon with four equal sides and four right angles. A rectangle has four right angles. 
Query: Is every square a rectangle?
"""},
    {"role": "assistant", "content": r"""
<reasoning_step> 
I will define the properties of a square based on the context. 
has_property(square, four_equal_sides). 
has_property(square, four_right_angles). 
is_a(square, polygon). 
</reasoning_step>

<reasoning_step> 
A rectangle is defined by having four right angles. 
is_a(Shape, rectangle) :- has_property(Shape, four_right_angles). 
</reasoning_step>

<reasoning_step> 
Now that the properties and definitions about squares, rectangles, as well as their angles and sides are established, I can query whether every square is a rectangle.
is_a(square, rectangle). 
</reasoning_step>

<database> 
has_property(square, four_equal_sides). 
has_property(square, four_right_angles). 
is_a(square, polygon). 
is_a(Shape, rectangle) :- has_property(Shape, four_right_angles). 
</database>

<query> 
is_a(square, rectangle). 
</query>

---
"""},
    
    # Example 4: Recursive Logic
    {"role": "user", "content": """
Now, process the following:
Context: In a food web, grass is a producer. Grasshoppers eat grass. Frogs eat grasshoppers. Snakes eat frogs. 
Query: If the grass is contaminated, are the snakes affected?
"""},
    {"role": "assistant", "content": r"""
<reasoning_step> 
First, I will define the direct consumption relationships (the edges in the graph). 
eats(grasshopper, grass). 
eats(frog, grasshopper). 
eats(snake, frog). 
is_producer(grass). 
</reasoning_step>

<reasoning_step> 
The contamination affects a species if it eats something contaminated. This is a recursive relationship (bioaccumulation). 
is_contaminated(grass). 
% Fact from query premise 
affected(Species) :- is_contaminated(Species). 
affected(Predator) :- eats(Predator, Prey), affected(Prey). 
</reasoning_step>

<reasoning_step> 
Now that the properties and definitions about the animals, the food web, and the contamination are established, I can query whether snakes are affected.
affected(snake). 
</reasoning_step>

<database> 
eats(grasshopper, grass). 
eats(frog, grasshopper). 
eats(snake, frog). 
is_producer(grass). 
is_contaminated(grass). 
affected(Species) :- is_contaminated(Species). 
affected(Predator) :- eats(Predator, Prey), affected(Prey). 
</database>

<query> 
affected(snake). 
</query>

---
"""}
]

def generate_prolog_generation_prompt(user_question: str, retrieved_context: str, most_recent_error: str) -> str:
    prompt = f"""
Now, process the following:
Context:
{retrieved_context}

User Question: 
{user_question}
"""

    if most_recent_error:
        prompt += f"""
Your recent Prolog database and query have errors. Fix them based on the following error message:
{most_recent_error}
"""

    return prompt

def generate_explanation_prompt(user_question: str, retrieved_context: str, retrieved_values: str) -> str:
    prompt = f"""
Your Prolog database has been successfully generated and consulted. Now, provide a detailed, logical, and step-by-step human-readable explanation of the justification Prolog gave. It should serve as context for answering the user's question.
User Question:
{user_question}

Context:
{retrieved_context}

The values retrieved from the query are:
{retrieved_values}
"""

    return prompt

**STEP 1.1: Load the model directly using Ollama Library**

---



In [2]:
import ollama
ollama_messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            
        ] + few_shot_examples
def generate_with_ollama(user_question: None | str, retrieved_context: None | str, retrieved_values: None | str, most_recent_error: None | str, prolog: bool, explanation: bool) -> dict: 
    answer = dict()
    if prolog:
        ollama_messages.append({'role': 'user', 'content': generate_prolog_generation_prompt(user_question, retrieved_context, most_recent_error)})
    if explanation:
        ollama_messages.append({'role': 'user', 'content': generate_explanation_prompt(user_question, retrieved_context, retrieved_values)})
    response = ollama.chat(
        model='llama3:instruct',
        messages=ollama_messages,
        options={
            'temperature': 0  
        },
        logprobs=True     
    )
    
    try: 
        answer['text_answer'] = response['message']['content']
        answer['logprobs'] = response.get('logprobs', None)
        return answer
    except Exception as e:
        print(f"Error during Ollama interaction: {e}")
        return None

**STEP 1.2: Load the model directly using OpenAI library Wrapper**

---



In [3]:
from openai import OpenAI

openai_messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
        ] + few_shot_examples

# 1. Initialize the Client to hijack the local Ollama server
client = OpenAI(
            base_url='http://localhost:11434/v1',  # Points to your local machine
            api_key='ollama'                       # Required string, but ignored by Ollama
        )

def generate_with_openai(user_question: None | str, retrieved_context: None | str, retrieved_values: None | str, most_recent_error: None | str, prolog: bool, explanation: bool) -> dict:
    answer = dict()
    
    if prolog:
        openai_messages.append({'role': 'user', 'content': generate_prolog_generation_prompt(user_question, retrieved_context, most_recent_error)})
    if explanation:
        openai_messages.append({'role': 'user', 'content': generate_explanation_prompt(user_question, retrieved_context, retrieved_values)})
    try:
        response = client.chat.completions.create(
            model="llama3:instruct",      
            messages=openai_messages,
            temperature=0,      # 0 = Deterministic (Scientific standard)
            logprobs=True,      # Turn on confidence scores
            top_logprobs=5      # <--- THE KEY FEATURE: Get the top 5 alternatives per token
        )

        answer["text_answer"] = response.choices[0].message.content
        answer["logprobs"] = response.choices[0].logprobs.content
        return answer
    except Exception as e:
        print(f"Error during OpenAI client interaction: {e}")
        return None

**STEP 1.3: Declare which wrapper would load the model**

In [4]:
WRAPPER = "OPENAI"

def generate(user_question: None | str, retrieved_context: None | str, retrieved_values: None | str, most_recent_error: None | str, prolog: bool, explanation: bool) -> dict:
    if prolog and explanation:
        raise ValueError("Both 'prolog' and 'explanation' cannot be True simultaneously.")
    if not(prolog or explanation):
        raise ValueError("Either 'prolog' or 'explanation' must be True.")
    if WRAPPER == "OLLAMA":
        return generate_with_ollama(user_question, retrieved_context, retrieved_values, most_recent_error, prolog, explanation)
    elif WRAPPER == "OPENAI":
        return generate_with_openai(user_question, retrieved_context, retrieved_values, most_recent_error, prolog, explanation)
    else:
        raise ValueError("Invalid WRAPPER specified. Choose either 'OLLAMA' or 'OPENAI'.")

**STEP 2: Create a function that uses regular expressions to capture the Prolog database and query in the LLM's response**

In [5]:
def capture_db_and_query(answer_text: str) -> tuple[str, str]:
    import re

    db_match = re.search(r"<database>(.*?)</database>", answer_text, re.DOTALL)
    # if "</query>" not in answer_text:
    #     answer_text += "</query>"
    query_match = re.search(r"<query>(.*?)</query>", answer_text, re.DOTALL)

    try:
        db = db_match.group(1)
    except:
        raise ValueError(f"Database not found in LLM's response \n{answer_text}")
    try:
        query = query_match.group(1)
    except:
        raise ValueError(f"Query not found in LLM's response: \n{answer_text}")

    return {"database":db, "query":query}

**STEP 3.1: Define a function that would ask the LLM to generate the database and the query**

___

In [6]:
import janus_swi as janus

def generate_prolog_code(user_question: str, retrieved_context: str, most_recent_error: None | str):
    for i in range(5):
        try:
            answer = generate(user_question=user_question, retrieved_context=retrieved_context, retrieved_values=None, most_recent_error=most_recent_error, prolog=True, explanation=False)
            database, query = capture_db_and_query(answer['text_answer'])["database"], capture_db_and_query(answer['text_answer'])["query"]
            print(f"Iteration {i} | Extracted Database:\n{database}\n")
            print(f"Iteration {i} | Extracted Query:\n{query}\n")
            janus.query_once("unload_file(user)")
            janus.consult("user", database)
            janus.query_once(query)
            most_recent_error = None
            print(f"Iteration {i} | Prolog code generation success")
            return answer
        except Exception as e:
            print(f"Iteration {i} | Error during prompt generation or model interaction: {e}")
            continue
        except janus.PrologError as e:
            print(f"Iteration {i} | Prolog Error: {e}")
            most_recent_error = str(e + " Ensure all Prolog syntax rules are followed. Enclose the final database within <database> and </database> tags, and the query within <query> and </query> tags.")
            print(most_recent_error)
            continue
        
    if most_recent_error:
        raise Exception("Failed to generate valid Prolog code after multiple attempts.")

**STEP 3.2: Define a function that would ask the LLM to generate a human-readable explanation for the Prolog reasoning trace**

In [7]:
def generate_explanation(user_question: str, retrieved_context: str, retrieved_values: None | str,most_recent_error: None | str):
    try:
        answer = generate(user_question=user_question, retrieved_context=retrieved_context, retrieved_values=None, most_recent_error=most_recent_error, prolog=False, explanation=True)
        return answer
    except Exception as e:
        print(f"Error during prompt generation or model interaction: {e}")

**STEP 4: Ask the LLM to generate the database and the query**

In [8]:
user_question = "Based on the internal records, who is the uncle of Leo?"
retrieved_context = """
A recent internal audit of Apex Corp revealed the following organizational and personal ties: 
1. Mark serves as the Chief Executive Officer of the company and has one sibling, a brother named Steven. 
2. Separately, payroll records indicate that Steven is the primary guardian and father of a child named Leo.
"""
most_recent_error = None

code = generate_prolog_code(user_question=user_question, retrieved_context=retrieved_context, most_recent_error=most_recent_error)
print(f"LLM's response:\n{code['text_answer']}\n")

Iteration 0 | Error during prompt generation or model interaction: Query not found in LLM's response: 
<reasoning_step>
First, I will define the family relationships based on the context.
parent(mark, steven). % Mark is the parent of Steven
sibling(mark, steven). % Mark and Steven are siblings
father(steven, leo). % Steven is the father of Leo
</reasoning_step>

<reasoning_step>
To answer the question about who is the uncle of Leo, I need a rule defining an uncle as the sibling of a parent.
uncle(Uncle, Child) :- sibling(Uncle, Parent), father(Parent, Child).
</reasoning_step>

<reasoning_step>
Now that the family relationships and the definition of an uncle are established, I can query who is the uncle of Leo.
uncle(mark, leo). % Mark is the uncle of Leo
</reasoning_step>

<database>
parent(mark, steven).
sibling(mark, steven).
father(steven, leo).
uncle(Uncle, Child) :- sibling(Uncle, Parent), father(Parent, Child).
</database>

<query>
uncle(mark, leo).
Iteration 1 | Extracted Datab

**STEP 5: Install s(CASP)**

In [9]:
import janus_swi as janus

list(janus.query("pack_install(scasp, [interactive(false)])."))
list(janus.query("use_module(library(scasp))."))
janus.query_once("use_module(library(scasp/human)).")

% Contacting server at https://www.swi-prolog.org/pack/query ... ok
% Pack scasp is up-to-date


{'truth': True}

**STEP 6: Define a dynamic scasp wrapper generator**

In [10]:
def generate_safe_scasp_wrapper(predicate_name, args_list):
    args_str = ", ".join(args_list)
    wrapper_name = f"explain"
    
    return rf"""
{wrapper_name}({args_str}, Explanation) :-
    % 1. Run the solver
    scasp({predicate_name}({args_str}), [tree(Tree)]),
    
    % 2. Capture the human-readable output into a string
    % We call human_justification_tree INSIDE with_output_to
    with_output_to(string(Explanation), 
        human_justification_tree(Tree, [tree_style(human)])
    ).
"""

**STEP 6.1: Test with sample database and query**

In [ ]:
logic = r"""
ceo(apex_corp, mark).
sibling(mark, steven).
parent(steven, leo).
guardian(steven, leo).

uncle(Uncle, Nephew) :- 
    parent(Parent, Nephew), 
    sibling(Uncle, Parent).
"""

wrapper = generate_safe_scasp_wrapper("uncle", ["Uncle", "Nephew"])
print(wrapper)
janus.consult("user", logic + "\n" + wrapper)

# The query asks for the variable 'Uncle' and the 'Tree'
query = r"""
explain_uncle(Uncle, leo, Explanation).
"""

try:
    result = janus.query_once(query)
    
    if result:
        print(f"Uncle Found: {result['Uncle']}")
        # result['Tree'] will be a nested list/dictionary structure
        print("\n--- Human-Readable Explanation ---")
        print(result['Explanation'])
    else:
        print("No solution found.")
except Exception as e:
    print(f"Error: {e}")

**STEP 7: Test the LLM-generated database and query**

In [11]:
import re
capture = capture_db_and_query(code['text_answer'])
logic = capture["database"]
print("Extracted Logic:\n", logic)
query = capture["query"]
print("Extracted Query:\n", query)
predicate_match = re.search(r"(\w+)\s*\(", query)
try:
    predicate = predicate_match.group(1)
except:
    raise ValueError("Could not extract predicate name from query")

arguments_match = re.search(r"\((.*?)\)", query)
try:
    arguments = arguments_match.group(1).replace(" ", "").split(",")
    print("Extracted Arguments:", arguments)
except:
    raise ValueError("Could not extract arguments from query")

wrapper = generate_safe_scasp_wrapper(predicate, arguments)
print("Generated Wrapper:\n", wrapper)
final_query = f"explain({', '.join(arguments)}, Explanation)."
print("Final Query:\n", final_query)
janus.consult("user", logic + "\n" + wrapper)
try:
    result = janus.query_once(final_query)
    retrieved_values = str()
    if result:
        for arg in arguments:  
            if arg.isupper():  # Check if it's a variable 
                print(f"{arg} Found: {result[arg]}")    
                retrieved_values += f"{arg}: {result[arg]} "
        # result['Tree'] will be a nested list/dictionary structure
        print("\n--- Human-Readable Explanation ---")
        print(result['Explanation'])
    else:
        print("No solution found.")
except Exception as e:
    print(f"Error: {e}")
    

        

Extracted Logic:
 
sibling(mark, steven). 
parent(steven, leo). 
uncle(Uncle, Child) :- sibling(Uncle, Parent), parent(Parent, Child).

Extracted Query:
 
uncle(mark, leo).

Extracted Arguments: ['mark', 'leo']
Generated Wrapper:
 
explain(mark, leo, Explanation) :-
    % 1. Run the solver
    scasp(uncle(mark, leo), [tree(Tree)]),

    % 2. Capture the human-readable output into a string
    % We call human_justification_tree INSIDE with_output_to
    with_output_to(string(Explanation), 
        human_justification_tree(Tree, [tree_style(human)])
    ).

Final Query:
 explain(mark, leo, Explanation).

--- Human-Readable Explanation ---
   uncle holds for mark, and leo, because
      sibling holds for mark, and steven, and
      parent holds for steven, and leo




**STEP 8: Ask the LLM to translate the human-readable explanation into a chain of logical reasoning**

In [12]:
def decode(user_question: str, human_readable_explanation: str, retrieved_values) -> str:
    explanation = generate(user_question=user_question, retrieved_context=human_readable_explanation, most_recent_error=None,retrieved_values=retrieved_values, prolog=False, explanation=True)
    try:
        return explanation['text_answer']
    except Exception as e:
        print(f"Error during explanation generation: {e}")
        return None
    
print(decode(user_question, result['Explanation'], retrieved_values))

<reasoning_step> 
To answer the question about who is the uncle of Leo, I will start by defining the relationships between Mark, Steven, and Leo. 
sibling(mark, steven). 
parent(steven, leo). 
</reasoning_step>

<reasoning_step> 
The context states that Mark has a brother named Steven, which means they share a sibling relationship. This is defined as "uncle" in the context of this problem.
uncle(mark, leo) :- sibling(mark, steven), parent(steven, leo). 
</reasoning_step>

<database> 
sibling(mark, steven). 
parent(steven, leo). 
uncle(mark, leo) :- sibling(mark, steven), parent(steven, leo). 
</database>

<query> 
uncle(mark, leo). 
</query>

---
The values retrieved from the query are:

uncle(mark, leo)

This means that based on the internal records, Mark is the uncle of Leo.
